In [1]:
from utils.utils import set_seed
import numpy as np
import torch
set_seed(1)

from torch_geometric.datasets import TUDataset
dataset = TUDataset(root="/tmp/PTC_MR/", name="PTC_MR")

print(dataset)
print(f"number of graphs: {len(dataset)}")
print(f"number of classes: {dataset.num_classes}")
print(f"number of node features: {dataset.num_node_features}")
print(f"number of edge features: {dataset.num_edge_features}")

c:\Users\User\Desktop\Maths\Projects\HodgeAndSheaf\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Seed fixada em 1
PTC_MR(344)
number of graphs: 344
number of classes: 2
number of node features: 18
number of edge features: 4


In [2]:
from utils.utils import create_data_splits

fold_loaders = create_data_splits(dataset, n_splits=4, batch_size=32)


Fold 1:
  Treino: 206 grafos
  Validação: 52 grafos
  Teste: 86 grafos


In [3]:
class GraphClassificationTrainer:
    def __init__(self, model, fold_loaders, config, device='cpu'):
        self.model = model.to(device)
        self.fold_loaders = fold_loaders
        self.config = config
        self.device = device
        
        # Configurando otimizador
        if config['optimizer'] == 'adam':
            self.optimizer = torch.optim.Adam(
                model.parameters(),
                lr=config['lr'],
                weight_decay=config['weight_decay']
            )
        
        # Scheduler de learning rate
        if config.get('scheduler') == 'step':
            self.scheduler = torch.optim.lr_scheduler.StepLR(
                self.optimizer,
                step_size=config['step_size'],
                gamma=config['gamma']
            )
        elif config.get('scheduler') == 'cosine':
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=config['epochs']
            )
        else:
            self.scheduler = None
        
        self.criterion = torch.nn.CrossEntropyLoss()
        
        # Para early stopping
        self.best_val_acc = 0
        self.patience_counter = 0
        self.best_model_state = None
    
    def train_epoch(self, train_loader):
        """
        Treina por uma época completa, iterando sobre todos os batches.
        """
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch in train_loader:
            batch = batch.to(self.device)
            
            self.optimizer.zero_grad()
            
            # Forward pass - ADICIONAR edge_attr! ✅
            out = self.model(batch.x, batch.edge_index, batch.batch,
                           edge_attr=batch.edge_attr if hasattr(batch, 'edge_attr') else None)
            
            loss = self.criterion(out, batch.y)
            loss.backward()
            
            if self.config.get('clip_grad'):
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.config['clip_grad']
                )
            
            self.optimizer.step()
            
            total_loss += loss.item() * batch.num_graphs
            pred = out.argmax(dim=1)
            correct += (pred == batch.y).sum().item()
            total += batch.num_graphs
        
        avg_loss = total_loss / total
        accuracy = correct / total
        
        return avg_loss, accuracy
    
    @torch.no_grad()
    def evaluate(self, loader):
        """
        Avalia o modelo em um DataLoader específico.
        """
        self.model.eval()
        correct = 0
        total = 0
        
        for batch in loader:
            batch = batch.to(self.device)
            
            # ADICIONAR edge_attr! ✅
            out = self.model(batch.x, batch.edge_index, batch.batch,
                           edge_attr=batch.edge_attr if hasattr(batch, 'edge_attr') else None)
            
            pred = out.argmax(dim=1)
            correct += (pred == batch.y).sum().item()
            total += batch.num_graphs
        
        return correct / total
    
    def train_fold(self, fold_idx):
        """
        Treina o modelo em um fold específico da validação cruzada.
        """
        print(f'\n=== Treinando Fold {fold_idx + 1} ===')
        
        # Reseta o modelo para cada fold
        self.model.apply(self._weight_reset)
        
        # Reseta otimizador e scheduler
        self.__init__(self.model, self.fold_loaders, self.config, self.device)
        
        loaders = self.fold_loaders[fold_idx]
        train_loader = loaders['train']
        val_loader = loaders['val']
        test_loader = loaders['test']
        
        history = {
            'train_loss': [],
            'train_acc': [],
            'val_acc': [],
            'test_acc': []
        }
        
        self.best_val_acc = 0
        self.patience_counter = 0
        
        for epoch in range(1, self.config['epochs'] + 1):
            # Treina por uma época
            train_loss, train_acc = self.train_epoch(train_loader)
            
            # Avalia em validação e teste
            val_acc = self.evaluate(val_loader)
            test_acc = self.evaluate(test_loader)
            
            # Guarda histórico
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)
            history['test_acc'].append(test_acc)
            
            # Atualiza learning rate
            if self.scheduler is not None:
                self.scheduler.step()
            
            # Early stopping
            if self.config.get('early_stopping'):
                if val_acc > self.best_val_acc:
                    self.best_val_acc = val_acc
                    self.best_model_state = self.model.state_dict().copy()
                    self.patience_counter = 0
                else:
                    self.patience_counter += 1
                
                if self.patience_counter >= self.config['patience']:
                    print(f'Early stopping na época {epoch}')
                    self.model.load_state_dict(self.best_model_state)
                    break
            
            # Log periódico
            if epoch % self.config['log_interval'] == 0:
                print(f'Época {epoch:03d}: Loss={train_loss:.4f}, '
                      f'Train={train_acc:.4f}, Val={val_acc:.4f}, Test={test_acc:.4f}')
        
        # Avaliação final no melhor modelo
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
        
        final_val_acc = self.evaluate(val_loader)
        final_test_acc = self.evaluate(test_loader)
        
        print(f'Melhor Val Acc: {final_val_acc:.4f}')
        print(f'Test Acc final: {final_test_acc:.4f}')
        
        return history, final_val_acc, final_test_acc
    
    def train_all_folds(self):
        """
        Executa validação cruzada completa em todos os folds.
        """
        val_accs = []
        test_accs = []
        all_histories = []
        
        for fold_idx in range(len(self.fold_loaders)):
            history, val_acc, test_acc = self.train_fold(fold_idx)
            val_accs.append(val_acc)
            test_accs.append(test_acc)
            all_histories.append(history)
        
        # Calcula estatísticas agregadas
        mean_val_acc = np.mean(val_accs)
        std_val_acc = np.std(val_accs)
        mean_test_acc = np.mean(test_accs)
        std_test_acc = np.std(test_accs)
        
        print('\n' + '='*50)
        print('RESULTADOS DA VALIDAÇÃO CRUZADA')
        print('='*50)
        print(f'Validação: {mean_val_acc:.4f} ± {std_val_acc:.4f}')
        print(f'Teste: {mean_test_acc:.4f} ± {std_test_acc:.4f}')
        print('='*50)
        
        return {
            'val_accs': val_accs,
            'test_accs': test_accs,
            'mean_val_acc': mean_val_acc,
            'std_val_acc': std_val_acc,
            'mean_test_acc': mean_test_acc,
            'std_test_acc': std_test_acc,
            'histories': all_histories
        }
    
    @staticmethod
    def _weight_reset(m):
        """
        Reseta os pesos de uma camada para reinicialização.
        """
        if hasattr(m, 'reset_parameters'):
            m.reset_parameters()

# Modelo

In [4]:
def compute_hodge_operators(edge_index, num_nodes):
    """
    Boundary Operators
    B1: Incidence Matrix
    L0: Laplaciano do Grafo
    L1: Laplaciano das Arestas
    """
    num_edges = edge_index.shape[1]
    device = edge_index.device
    B1 = torch.zeros(num_nodes, num_edges, device=device)
    
    for edge_idx in range(num_edges):
        source = edge_index[0, edge_idx]  # Nó de origem
        target = edge_index[1, edge_idx]  # Nó de destino
        
        B1[source, edge_idx] = 1   # Aresta SAI (+1)
        B1[target, edge_idx] = -1  # Aresta CHEGA (-1)
    
    # === Computar Laplacianos ===
    L0 = B1 @ B1.T  # Graph Laplacian (node space)
    L1 = B1.T @ B1  # Hodge Laplacian (edge space)
    
    return L0, L1, B1

def normalize_laplacian(L):
    """ 
    L_norm = D^(-1/2) @ L @ D^(-1/2)
    """
    # Grau: soma dos valores absolutos em cada linha
    degree = torch.abs(L).sum(dim=1).clamp(min=1e-6)  # Evita divisão por zero
    
    # D^(-1/2)
    deg_inv_sqrt = torch.pow(degree, -0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.  # Remove infinitos
    
    # L_norm = D^(-1/2) @ L @ D^(-1/2)
    L_norm = deg_inv_sqrt.unsqueeze(1) * L * deg_inv_sqrt.unsqueeze(0)
    
    return L_norm

In [5]:
test_edge_index_2 = torch.tensor([
    [0, 0, 1, 2],  # source
    [1, 2, 2, 0]   # target
])

L0_2, L1_2, B1_2 = compute_hodge_operators(test_edge_index_2, num_nodes=3)

print(f"📐 Dimensões (Grafo 2):")
print(f"   B₁: {B1_2.shape} (nós × arestas)")
print(f"   L₀: {L0_2.shape} (nós × nós)")
print(f"   L₁: {L1_2.shape} (arestas × arestas)")

print(f"\n📋 Matriz B₁:")
print(B1_2)

print(f"\n📋 L₀ (Graph Laplacian) - [3×3]:")
print(L0_2)

print(f"\n📋 L₁ (Hodge Laplacian) - [4×4]:")
print(L1_2)

print("\n✅ Agora L₀ e L₁ têm dimensões diferentes!")

📐 Dimensões (Grafo 2):
   B₁: torch.Size([3, 4]) (nós × arestas)
   L₀: torch.Size([3, 3]) (nós × nós)
   L₁: torch.Size([4, 4]) (arestas × arestas)

📋 Matriz B₁:
tensor([[ 1.,  1.,  0., -1.],
        [-1.,  0.,  1.,  0.],
        [ 0., -1., -1.,  1.]])

📋 L₀ (Graph Laplacian) - [3×3]:
tensor([[ 3., -1., -2.],
        [-1.,  2., -1.],
        [-2., -1.,  3.]])

📋 L₁ (Hodge Laplacian) - [4×4]:
tensor([[ 2.,  1., -1., -1.],
        [ 1.,  2.,  1., -2.],
        [-1.,  1.,  2., -1.],
        [-1., -2., -1.,  2.]])

✅ Agora L₀ e L₁ têm dimensões diferentes!


In [6]:
from torch_geometric.nn import global_mean_pool, global_max_pool, global_add_pool
import torch.nn as nn
import torch.nn.functional as F
class DualMessagePassingHodgeGNN(nn.Module):
    """
    Dual Message Passing Hodge GNN
    
    Arquitetura:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Em cada layer:
    
    1. WITHIN-DOMAIN AGGREGATION:
       - Nós agregam de nós vizinhos (via L₀)
       - Arestas agregam de arestas vizinhas (via L₁)
    
    2. CROSS-DOMAIN MESSAGE PASSING:
       - Nós → Arestas (via B₁)
       - Arestas → Nós (via B₁ᵀ)
    
    3. UPDATE:
       - BatchNorm + ReLU + Dropout
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    Isso permite que informação flua tanto dentro de cada
    espaço quanto ENTRE espaços!
    """
    
    def __init__(self, 
                 num_node_features,
                 num_edge_features=0,
                 num_classes=2,
                 hidden_dim=64,
                 num_layers=3,
                 pooling='mean',
                 dropout=0.5,
                 normalize_L=True):
        """
        Args:
            num_node_features: Dimensão das node features (7 para MUTAG)
            num_edge_features: Dimensão das edge features (4 para MUTAG)
            num_classes: Número de classes
            hidden_dim: Dimensão das camadas ocultas
            num_layers: Número de camadas de message passing
            pooling: 'mean', 'max', ou 'sum'
            dropout: Taxa de dropout
            normalize_L: Se deve normalizar L₀ e L₁
        """
        super().__init__()
        
        self.num_layers = num_layers
        self.pooling = pooling
        self.dropout = dropout
        self.normalize_L = normalize_L
        
        print("🏗️  Construindo DualMessagePassingHodgeGNN...")
        print(f"   Node features: {num_node_features}")
        print(f"   Edge features: {num_edge_features}")
        print(f"   Hidden dim: {hidden_dim}")
        print(f"   Num layers: {num_layers}")
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # NODE PROCESSING (L₀ space)
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        self.node_linears = nn.ModuleList()
        for i in range(num_layers):
            in_dim = num_node_features if i == 0 else hidden_dim
            self.node_linears.append(nn.Linear(in_dim, hidden_dim))
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # EDGE PROCESSING (L₁ space)
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        self.edge_linears = nn.ModuleList()
        for i in range(num_layers):
            if i == 0:
                # Primeira camada: concatenação de node features + edge_attr
                in_dim = 2 * num_node_features + num_edge_features
            else:
                in_dim = hidden_dim
            self.edge_linears.append(nn.Linear(in_dim, hidden_dim))
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # CROSS-DOMAIN MESSAGE PASSING
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        # Nós → Arestas
        self.node_to_edge = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim)
            for _ in range(num_layers)
        ])
        
        # Arestas → Nós
        self.edge_to_node = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim)
            for _ in range(num_layers)
        ])
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # NORMALIZATION LAYERS
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        self.node_norms = nn.ModuleList([
            nn.BatchNorm1d(hidden_dim) 
            for _ in range(num_layers)
        ])
        
        self.edge_norms = nn.ModuleList([
            nn.BatchNorm1d(hidden_dim) 
            for _ in range(num_layers)
        ])
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # FUSION & CLASSIFICATION
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        self.classifier = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),  # Concatena node + edge
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
        # Contar parâmetros
        total_params = sum(p.numel() for p in self.parameters())
        print(f"   Total de parâmetros: {total_params:,}")
        print("✅ Modelo construído!\n")
    
    def forward(self, x, edge_index, batch, edge_attr=None):
        """
        Forward pass com Dual Message Passing.
        
        Args:
            x: [num_nodes, num_node_features]
            edge_index: [2, num_edges]
            batch: [num_nodes] - indica qual nó pertence a qual grafo
            edge_attr: [num_edges, num_edge_features] (opcional)
        
        Returns:
            logits: [batch_size, num_classes]
        """
        num_nodes = x.shape[0]
        num_edges = edge_index.shape[1]
        device = x.device
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 1: Computar Operadores Hodge
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        L0, L1, B1 = compute_hodge_operators(edge_index, num_nodes)
        
        # Normalizar Laplacians
        if self.normalize_L:
            L0 = normalize_laplacian(L0)
            L1 = normalize_laplacian(L1)
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 2: Inicializar Features
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        # Node features (já temos)
        h_node = x  # [num_nodes, num_node_features]
        
        # Edge features: concatenar features dos nós nas pontas
        h_edge = torch.cat([
            x[edge_index[0]],  # Source node features
            x[edge_index[1]]   # Target node features
        ], dim=1)  # [num_edges, 2*num_node_features]
        
        # Adicionar edge_attr se disponível
        if edge_attr is not None:
            h_edge = torch.cat([h_edge, edge_attr.float()], dim=1)
            # [num_edges, 2*num_node_features + num_edge_features]
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 3: Message Passing Layers
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        for layer in range(self.num_layers):
            # ┌─────────────────────────────────────────┐
            # │ 3.1: WITHIN-DOMAIN AGGREGATION          │
            # └─────────────────────────────────────────┘
            
            # Nós agregam de nós vizinhos (via L₀)
            h_node_agg = L0 @ h_node  # [num_nodes, hidden_dim]
            h_node_new = self.node_linears[layer](h_node_agg)
            
            # Arestas agregam de arestas vizinhas (via L₁)
            h_edge_agg = L1 @ h_edge  # [num_edges, hidden_dim]
            h_edge_new = self.edge_linears[layer](h_edge_agg)
            
            # ┌─────────────────────────────────────────┐
            # │ 3.2: CROSS-DOMAIN MESSAGE PASSING       │
            # └─────────────────────────────────────────┘
            
            # Nós → Arestas (via estrutura do grafo)
            # Cada aresta (u,v) recebe mensagem dos nós u e v
            node_msg = self.node_to_edge[layer](h_node_new)
            edge_receives_from_nodes = (
                node_msg[edge_index[0]] +  # Mensagem do source
                node_msg[edge_index[1]]    # Mensagem do target
            ) / 2  # Média
            h_edge_new = h_edge_new + edge_receives_from_nodes
            
            # Arestas → Nós (via B₁ᵀ)
            # Cada nó agrega mensagens de suas arestas incidentes
            edge_msg = self.edge_to_node[layer](h_edge_new)
            node_receives_from_edges = B1 @ edge_msg  # B₁: [num_nodes, num_edges]
            h_node_new = h_node_new + node_receives_from_edges
            
            # ┌─────────────────────────────────────────┐
            # │ 3.3: NORMALIZATION & ACTIVATION         │
            # └─────────────────────────────────────────┘
            
            # Nodes
            h_node_new = self.node_norms[layer](h_node_new)
            h_node_new = F.relu(h_node_new)
            h_node_new = F.dropout(h_node_new, p=self.dropout, training=self.training)
            
            # Edges
            h_edge_new = self.edge_norms[layer](h_edge_new)
            h_edge_new = F.relu(h_edge_new)
            h_edge_new = F.dropout(h_edge_new, p=self.dropout, training=self.training)
            
            # Update para próxima iteração
            h_node = h_node_new
            h_edge = h_edge_new
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 4: Graph-Level Pooling
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        # Node pooling
        if self.pooling == 'mean':
            h_node_graph = global_mean_pool(h_node, batch)
        elif self.pooling == 'max':
            h_node_graph = global_max_pool(h_node, batch)
        elif self.pooling == 'sum':
            h_node_graph = global_add_pool(h_node, batch)
        
        # Edge pooling
        edge_batch = batch[edge_index[0]]  # Qual grafo cada aresta pertence
        if self.pooling == 'mean':
            h_edge_graph = global_mean_pool(h_edge, edge_batch)
        elif self.pooling == 'max':
            h_edge_graph = global_max_pool(h_edge, edge_batch)
        elif self.pooling == 'sum':
            h_edge_graph = global_add_pool(h_edge, edge_batch)
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 5: Fusion & Classification
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        # Concatenar representações de nós e arestas
        h_graph = torch.cat([h_node_graph, h_edge_graph], dim=1)
        # [batch_size, 2*hidden_dim]
        
        # Classificação
        logits = self.classifier(h_graph)
        # [batch_size, num_classes]
        
        return logits


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TESTE DO MODELO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("🧪 Testando o modelo...")

# Criar modelo
model = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,  # 7
    num_edge_features=dataset.num_edge_features,  # 4
    num_classes=dataset.num_classes,              # 2
    hidden_dim=64,
    num_layers=3,
    pooling='max',
    dropout=0
)

# Testar com um batch
test_batch = next(iter(fold_loaders[0]['train']))
print(f"\n📦 Batch de teste:")
print(f"   Num grafos: {test_batch.num_graphs}")
print(f"   Num nós: {test_batch.num_nodes}")
print(f"   Num arestas: {test_batch.num_edges}")

# Forward pass
with torch.no_grad():
    logits = model(test_batch.x, test_batch.edge_index, test_batch.batch, test_batch.edge_attr)

print(f"\n✅ Forward pass bem-sucedido!")
print(f"   Output shape: {logits.shape}")  # [batch_size, num_classes]
print(f"   Output (primeiros 3): {logits[:3]}")

🧪 Testando o modelo...
🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features: 4
   Hidden dim: 64
   Num layers: 3
   Total de parâmetros: 54,594
✅ Modelo construído!


📦 Batch de teste:
   Num grafos: 32
   Num nós: 442
   Num arestas: 896

✅ Forward pass bem-sucedido!
   Output shape: torch.Size([32, 2])
   Output (primeiros 3): tensor([[ 0.4178,  0.1667],
        [-0.0027,  0.2612],
        [ 0.1635,  0.0128]])


In [7]:
# Configuração de treinamento
config = {
    'optimizer': 'adam',
    'lr': 1e-4,
    'weight_decay': 0,
    'scheduler': 'step',
    'step_size': 50,
    'gamma': 0.5,
    'epochs': 250,
    # 'clip_grad': 1.0,
    # 'early_stopping': True,
    # 'patience': 50,
    'log_interval': 10
}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando device: {device}')
# Treinar com seu trainer existente!
trainer = GraphClassificationTrainer(model, fold_loaders, config, device=device)
results = trainer.train_all_folds()

print("\n Resultados HodgeGNN:")
print(f"Validação: {results['mean_val_acc']:.4f} ± {results['std_val_acc']:.4f}")
print(f"Teste: {results['mean_test_acc']:.4f} ± {results['std_test_acc']:.4f}")

Usando device: cpu

=== Treinando Fold 1 ===
Época 010: Loss=0.6202, Train=0.6845, Val=0.6154, Test=0.6163
Época 020: Loss=0.5834, Train=0.7233, Val=0.6538, Test=0.6163
Época 030: Loss=0.5388, Train=0.7524, Val=0.7115, Test=0.6047
Época 040: Loss=0.5156, Train=0.7670, Val=0.6923, Test=0.5581
Época 050: Loss=0.4833, Train=0.7427, Val=0.7115, Test=0.6047
Época 060: Loss=0.4637, Train=0.7913, Val=0.6923, Test=0.5814
Época 070: Loss=0.4660, Train=0.8155, Val=0.6731, Test=0.6047
Época 080: Loss=0.4495, Train=0.7767, Val=0.6538, Test=0.5930
Época 090: Loss=0.4268, Train=0.8204, Val=0.6923, Test=0.6395
Época 100: Loss=0.4135, Train=0.8301, Val=0.6923, Test=0.5698
Época 110: Loss=0.4244, Train=0.8350, Val=0.6731, Test=0.6163
Época 120: Loss=0.3850, Train=0.8544, Val=0.7115, Test=0.5814
Época 130: Loss=0.3986, Train=0.8592, Val=0.6731, Test=0.5698
Época 140: Loss=0.3749, Train=0.8641, Val=0.6923, Test=0.5814
Época 150: Loss=0.3726, Train=0.8689, Val=0.7115, Test=0.5930
Época 160: Loss=0.3754, T

In [8]:
from torch_geometric.nn import global_mean_pool, global_max_pool, global_add_pool

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# HODGE CONVOLUTIONAL LAYER (Reutilizável)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class HodgeConvLayer(nn.Module):
    """
    Camada de convolução via Hodge Laplacian.
    
    Funciona tanto para L₀ (nós) quanto L₁ (arestas)!
    
    Operação: L @ features → Linear → BatchNorm → ReLU → Dropout
    """
    def __init__(self, in_features, out_features, 
                 batch_norm=True, dropout=0.0):
        super().__init__()
        
        self.linear = nn.Linear(in_features, out_features)
        self.batch_norm = nn.BatchNorm1d(out_features) if batch_norm else None
        self.dropout = dropout
        
    def forward(self, features, L):
        """
        Args:
            features: [N, in_features] - features (nós ou arestas)
            L: [N, N] - Laplacian (L₀ ou L₁)
        
        Returns:
            out: [N, out_features]
        """
        # 1. Convolução: agregar de vizinhos via Laplacian
        aggregated = L @ features
        
        # 2. Transformação linear
        out = self.linear(aggregated)
        
        # 3. Batch normalization
        if self.batch_norm is not None:
            out = self.batch_norm(out)
        
        # 4. Ativação
        out = F.relu(out)
        
        # 5. Dropout
        out = F.dropout(out, p=self.dropout, training=self.training)
        
        return out


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DUAL MESSAGE PASSING HODGE GNN (Versão Flexível)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class DualMessagePassingHodgeGNN(nn.Module):
    """
    Dual Message Passing Hodge GNN - Versão Totalmente Configurável
    
    Features:
    ✅ Controle total das dimensões de cada camada convolucional
    ✅ MLP customizável no final
    ✅ Usa HodgeConvLayer modular
    ✅ Cross-domain message passing entre L₀ e L₁
    """
    
    def __init__(self, 
                 num_node_features,
                 num_edge_features=0,
                 num_classes=2,
                 node_hidden_dims=[64, 64, 32],      # ← Lista de dims para convs de nós!
                 edge_hidden_dims=[64, 64, 32],      # ← Lista de dims para convs de arestas!
                 mlp_hidden_dims=[128, 64],          # ← Lista de dims para MLP final!
                 pooling='mean',
                 dropout=0.5,
                 batch_norm=True,
                 normalize_L=True):
        """
        Args:
            num_node_features: Dimensão das node features (7 para MUTAG)
            num_edge_features: Dimensão das edge features (4 para MUTAG)
            num_classes: Número de classes
            node_hidden_dims: Lista com dimensões das camadas de nós
                Ex: [32, 64, 32] = 3 camadas
            edge_hidden_dims: Lista com dimensões das camadas de arestas
                Ex: [32, 64, 32] = 3 camadas
            mlp_hidden_dims: Lista com dimensões do MLP classificador
                Ex: [128, 64] = MLP com 2 camadas ocultas
            pooling: 'mean', 'max', ou 'sum'
            dropout: Taxa de dropout
            batch_norm: Usar BatchNorm nas convoluções
            normalize_L: Normalizar L₀ e L₁
        """
        super().__init__()
        
        # Validação
        if len(node_hidden_dims) != len(edge_hidden_dims):
            raise ValueError("node_hidden_dims e edge_hidden_dims devem ter o mesmo tamanho!")
        
        self.num_layers = len(node_hidden_dims)
        self.pooling = pooling
        self.dropout = dropout
        self.normalize_L = normalize_L
        self.node_hidden_dims = node_hidden_dims
        self.edge_hidden_dims = edge_hidden_dims
        
        print("🏗️  Construindo DualMessagePassingHodgeGNN...")
        print(f"   Node features: {num_node_features}")
        print(f"   Edge features: {num_edge_features}")
        print(f"   Node conv dims: {node_hidden_dims}")
        print(f"   Edge conv dims: {edge_hidden_dims}")
        print(f"   MLP dims: {mlp_hidden_dims}")
        print(f"   Num layers: {self.num_layers}")
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # NODE CONVOLUTIONS (L₀ space)
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        self.node_convs = nn.ModuleList()
        
        for i, hidden_dim in enumerate(node_hidden_dims):
            in_dim = num_node_features if i == 0 else node_hidden_dims[i-1]
            self.node_convs.append(
                HodgeConvLayer(in_dim, hidden_dim, batch_norm, dropout)
            )
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # EDGE CONVOLUTIONS (L₁ space)
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        self.edge_convs = nn.ModuleList()
        
        for i, hidden_dim in enumerate(edge_hidden_dims):
            if i == 0:
                # Primeira camada: [x_u, x_v, edge_attr]
                in_dim = 2 * num_node_features + num_edge_features
            else:
                in_dim = edge_hidden_dims[i-1]
            
            self.edge_convs.append(
                HodgeConvLayer(in_dim, hidden_dim, batch_norm, dropout)
            )
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # CROSS-DOMAIN MESSAGE PASSING
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        self.node_to_edge = nn.ModuleList([
            nn.Linear(node_hidden_dims[i], edge_hidden_dims[i])
            for i in range(self.num_layers)
        ])
        
        self.edge_to_node = nn.ModuleList([
            nn.Linear(edge_hidden_dims[i], node_hidden_dims[i])
            for i in range(self.num_layers)
        ])
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # MLP CLASSIFIER (Customizável!)
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        mlp_layers = []
        
        # Input do MLP: concatenação de node e edge representations
        input_dim = node_hidden_dims[-1] + edge_hidden_dims[-1]
        
        # Camadas ocultas do MLP
        for mlp_dim in mlp_hidden_dims:
            mlp_layers.extend([
                nn.Linear(input_dim, mlp_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            input_dim = mlp_dim
        
        # Camada final: última hidden_dim → num_classes
        mlp_layers.append(nn.Linear(input_dim, num_classes))
        
        self.classifier = nn.Sequential(*mlp_layers)
        
        # Contar parâmetros
        total_params = sum(p.numel() for p in self.parameters())
        print(f"   Total de parâmetros: {total_params:,}")
        print("✅ Modelo construído!\n")
    
    def forward(self, x, edge_index, batch, edge_attr=None):
        """
        Forward pass com Dual Message Passing.
        
        Args:
            x: [num_nodes, num_node_features]
            edge_index: [2, num_edges]
            batch: [num_nodes]
            edge_attr: [num_edges, num_edge_features] (opcional)
        
        Returns:
            logits: [batch_size, num_classes]
        """
        num_nodes = x.shape[0]
        device = x.device
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 1: Computar Operadores Hodge
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        L0, L1, B1 = compute_hodge_operators(edge_index, num_nodes)
        
        if self.normalize_L:
            L0 = normalize_laplacian(L0)
            L1 = normalize_laplacian(L1)
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 2: Inicializar Features
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        h_node = x  # [num_nodes, num_node_features]
        
        # Edge features: [x_u, x_v]
        h_edge = torch.cat([
            x[edge_index[0]],  # Source features
            x[edge_index[1]]   # Target features
        ], dim=1)
        
        # Adicionar edge_attr se disponível
        if edge_attr is not None:
            h_edge = torch.cat([h_edge, edge_attr.float()], dim=1)
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 3: Message Passing Layers
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        for layer in range(self.num_layers):
            # ┌─────────────────────────────────────────┐
            # │ Within-Domain: Convoluções!             │
            # └─────────────────────────────────────────┘
            
            h_node_new = self.node_convs[layer](h_node, L0)
            h_edge_new = self.edge_convs[layer](h_edge, L1)
            
            # ┌─────────────────────────────────────────┐
            # │ Cross-Domain: Message Passing           │
            # └─────────────────────────────────────────┘
            
            # Nós → Arestas
            node_msg = self.node_to_edge[layer](h_node_new)
            edge_receives = (node_msg[edge_index[0]] + node_msg[edge_index[1]]) / 2
            h_edge_new = h_edge_new + edge_receives
            
            # Arestas → Nós
            edge_msg = self.edge_to_node[layer](h_edge_new)
            node_receives = B1 @ edge_msg
            h_node_new = h_node_new + node_receives
            
            # Update
            h_node = h_node_new
            h_edge = h_edge_new
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 4: Graph-Level Pooling
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        if self.pooling == 'mean':
            h_node_graph = global_mean_pool(h_node, batch)
        elif self.pooling == 'max':
            h_node_graph = global_max_pool(h_node, batch)
        elif self.pooling == 'sum':
            h_node_graph = global_add_pool(h_node, batch)
        
        edge_batch = batch[edge_index[0]]
        if self.pooling == 'mean':
            h_edge_graph = global_mean_pool(h_edge, edge_batch)
        elif self.pooling == 'max':
            h_edge_graph = global_max_pool(h_edge, edge_batch)
        elif self.pooling == 'sum':
            h_edge_graph = global_add_pool(h_edge, edge_batch)
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # STEP 5: Fusion & Classification
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        h_graph = torch.cat([h_node_graph, h_edge_graph], dim=1)
        logits = self.classifier(h_graph)
        
        return logits


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TESTES COM DIFERENTES CONFIGURAÇÕES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("="*60)
print("TESTANDO DIFERENTES ARQUITETURAS")
print("="*60)

# ━━━ Configuração 1: Simples ━━━
print("\n🔵 Config 1: Arquitetura Simples")
model1 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[32, 32],          # 2 camadas, dim 32
    edge_hidden_dims=[32, 32],
    mlp_hidden_dims=[64],               # 1 camada no MLP
    dropout=0.01
)

# ━━━ Configuração 2: Balanceada ━━━
print("\n🟢 Config 2: Arquitetura Balanceada")
model2 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[32, 64, 32],      # 3 camadas, "funil"
    edge_hidden_dims=[32, 64, 32],
    mlp_hidden_dims=[128, 64],          # 2 camadas no MLP
    dropout=0.01
)

# ━━━ Configuração 3: Profunda ━━━
print("\n🟠 Config 3: Arquitetura Profunda")
model3 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[64, 128, 64, 32], # 4 camadas!
    edge_hidden_dims=[64, 128, 64, 32],
    mlp_hidden_dims=[256, 128, 64],     # 3 camadas no MLP
    dropout=0.01
)

# ━━━ Configuração 4: Assimétrica (Node ≠ Edge) ━━━
print("\n🔴 Config 4: Node e Edge com dims diferentes")
model4 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[32, 64, 32],      # Nós: 3 camadas
    edge_hidden_dims=[64, 128, 64],     # Arestas: dims maiores!
    mlp_hidden_dims=[128, 64],
    dropout=0.01
)

model4 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[32, 32, 32, 32],      # Nós: 3 camadas
    edge_hidden_dims=[32, 32, 32, 32],     # Arestas: dims maiores!
    mlp_hidden_dims=[128, 128],
    dropout=0.01
)
# ━━━ Teste Forward ━━━
print("\n" + "="*60)
print("TESTANDO FORWARD PASS")
print("="*60)

test_batch = next(iter(fold_loaders[0]['train']))

for i, model in enumerate([model1, model2, model3, model4], 1):
    with torch.no_grad():
        logits = model(test_batch.x, test_batch.edge_index, 
                      test_batch.batch, test_batch.edge_attr)
    print(f"✅ Config {i}: Output shape = {logits.shape}")

print("\n🎉 Todos os modelos funcionando!")

TESTANDO DIFERENTES ARQUITETURAS

🔵 Config 1: Arquitetura Simples
🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features: 4
   Node conv dims: [32, 32]
   Edge conv dims: [32, 32]
   MLP dims: [64]
   Num layers: 2
   Total de parâmetros: 12,802
✅ Modelo construído!


🟢 Config 2: Arquitetura Balanceada
🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features: 4
   Node conv dims: [32, 64, 32]
   Edge conv dims: [32, 64, 32]
   MLP dims: [128, 64]
   Num layers: 3
   Total de parâmetros: 40,066
✅ Modelo construído!


🟠 Config 3: Arquitetura Profunda
🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features: 4
   Node conv dims: [64, 128, 64, 32]
   Edge conv dims: [64, 128, 64, 32]
   MLP dims: [256, 128, 64]
   Num layers: 4
   Total de parâmetros: 152,002
✅ Modelo construído!


🔴 Config 4: Node e Edge com dims diferentes
🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features:

In [9]:

trainer = GraphClassificationTrainer(model1, fold_loaders, config, device=device)
results = trainer.train_all_folds()


=== Treinando Fold 1 ===
Época 010: Loss=0.6776, Train=0.5728, Val=0.5769, Test=0.6047
Época 020: Loss=0.6721, Train=0.5922, Val=0.5962, Test=0.6047
Época 030: Loss=0.6639, Train=0.5971, Val=0.5962, Test=0.6047
Época 040: Loss=0.6565, Train=0.6117, Val=0.6154, Test=0.6047
Época 050: Loss=0.6541, Train=0.6262, Val=0.6731, Test=0.5930
Época 060: Loss=0.6434, Train=0.6165, Val=0.6538, Test=0.6047
Época 070: Loss=0.6497, Train=0.5971, Val=0.6538, Test=0.5930
Época 080: Loss=0.6413, Train=0.6359, Val=0.6731, Test=0.5698
Época 090: Loss=0.6420, Train=0.6505, Val=0.6731, Test=0.5698
Época 100: Loss=0.6396, Train=0.6456, Val=0.6731, Test=0.5581
Época 110: Loss=0.6310, Train=0.6359, Val=0.6731, Test=0.5581
Época 120: Loss=0.6317, Train=0.6165, Val=0.6731, Test=0.5581
Época 130: Loss=0.6354, Train=0.6456, Val=0.6538, Test=0.5814
Época 140: Loss=0.6285, Train=0.6505, Val=0.6731, Test=0.5814
Época 150: Loss=0.6344, Train=0.6505, Val=0.6538, Test=0.5698
Época 160: Loss=0.6321, Train=0.6553, Val=0.

In [10]:

trainer = GraphClassificationTrainer(model2, fold_loaders, config, device=device)
results = trainer.train_all_folds()


=== Treinando Fold 1 ===
Época 010: Loss=0.6817, Train=0.5534, Val=0.5962, Test=0.5698
Época 020: Loss=0.6694, Train=0.5874, Val=0.5769, Test=0.5698
Época 030: Loss=0.6655, Train=0.5728, Val=0.5962, Test=0.5930
Época 040: Loss=0.6522, Train=0.5922, Val=0.5962, Test=0.5930
Época 050: Loss=0.6395, Train=0.5971, Val=0.5962, Test=0.5465
Época 060: Loss=0.6330, Train=0.6408, Val=0.5577, Test=0.5349
Época 070: Loss=0.6295, Train=0.6408, Val=0.5385, Test=0.5465
Época 080: Loss=0.6169, Train=0.6456, Val=0.6346, Test=0.5465
Época 090: Loss=0.6186, Train=0.6990, Val=0.5385, Test=0.5465
Época 100: Loss=0.6088, Train=0.6845, Val=0.5192, Test=0.5349
Época 110: Loss=0.6072, Train=0.6553, Val=0.5385, Test=0.5349
Época 120: Loss=0.6108, Train=0.6408, Val=0.5192, Test=0.5349
Época 130: Loss=0.5972, Train=0.6650, Val=0.5192, Test=0.5349
Época 140: Loss=0.5958, Train=0.6699, Val=0.5192, Test=0.5465
Época 150: Loss=0.5925, Train=0.6893, Val=0.5192, Test=0.5581
Época 160: Loss=0.5935, Train=0.6748, Val=0.

In [11]:

trainer = GraphClassificationTrainer(model3, fold_loaders, config, device=device)
results = trainer.train_all_folds()


=== Treinando Fold 1 ===
Época 010: Loss=0.6813, Train=0.5583, Val=0.5577, Test=0.5581
Época 020: Loss=0.6697, Train=0.5825, Val=0.6154, Test=0.6047
Época 030: Loss=0.6442, Train=0.6068, Val=0.5769, Test=0.5465
Época 040: Loss=0.6421, Train=0.6796, Val=0.5577, Test=0.5349
Época 050: Loss=0.6169, Train=0.6117, Val=0.5962, Test=0.5930
Época 060: Loss=0.5972, Train=0.6456, Val=0.5962, Test=0.5930
Época 070: Loss=0.5861, Train=0.6845, Val=0.5577, Test=0.5465
Época 080: Loss=0.5727, Train=0.6505, Val=0.5769, Test=0.5814
Época 090: Loss=0.5631, Train=0.6990, Val=0.5962, Test=0.5930
Época 100: Loss=0.5516, Train=0.6893, Val=0.6154, Test=0.5349
Época 110: Loss=0.5549, Train=0.7087, Val=0.6346, Test=0.5698
Época 120: Loss=0.5454, Train=0.6893, Val=0.5962, Test=0.5698
Época 130: Loss=0.5429, Train=0.7039, Val=0.5962, Test=0.5349
Época 140: Loss=0.5402, Train=0.7039, Val=0.5769, Test=0.5465
Época 150: Loss=0.5158, Train=0.7379, Val=0.5962, Test=0.5814
Época 160: Loss=0.5275, Train=0.6796, Val=0.

In [12]:

trainer = GraphClassificationTrainer(model4, fold_loaders, config, device=device)
results = trainer.train_all_folds()


=== Treinando Fold 1 ===
Época 010: Loss=0.6817, Train=0.5583, Val=0.5577, Test=0.5581
Época 020: Loss=0.6711, Train=0.5583, Val=0.5577, Test=0.5581
Época 030: Loss=0.6629, Train=0.5777, Val=0.5962, Test=0.6047
Época 040: Loss=0.6554, Train=0.5777, Val=0.5577, Test=0.5814
Época 050: Loss=0.6487, Train=0.6505, Val=0.5962, Test=0.5349
Época 060: Loss=0.6323, Train=0.6214, Val=0.6154, Test=0.5698
Época 070: Loss=0.6261, Train=0.6553, Val=0.5192, Test=0.5698
Época 080: Loss=0.6188, Train=0.6942, Val=0.5577, Test=0.5814
Época 090: Loss=0.6155, Train=0.6165, Val=0.5000, Test=0.5698
Época 100: Loss=0.6098, Train=0.6650, Val=0.5000, Test=0.5698
Época 110: Loss=0.5997, Train=0.6650, Val=0.5577, Test=0.5814
Época 120: Loss=0.6027, Train=0.6553, Val=0.5192, Test=0.5930
Época 130: Loss=0.5998, Train=0.6553, Val=0.5385, Test=0.5698
Época 140: Loss=0.5997, Train=0.6845, Val=0.5385, Test=0.5930
Época 150: Loss=0.5918, Train=0.6262, Val=0.5385, Test=0.5698
Época 160: Loss=0.6014, Train=0.6505, Val=0.

In [15]:
model5 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[32, 32, 32, 32],      # Nós: 3 camadas
    edge_hidden_dims=[32, 32, 32, 32],     # Arestas: dims maiores!
    mlp_hidden_dims=[128, 128],
    dropout=0.01,
    pooling= "sum"
)
config = {
    'optimizer': 'adam',
    'lr': 1e-4,
    'weight_decay': 0,
    'scheduler': 'step',
    'step_size': 50,
    'gamma': 0.5,
    'epochs': 100,
    # 'clip_grad': 1.0,
    # 'early_stopping': True,
    # 'patience': 50,
    'log_interval': 10
}

trainer = GraphClassificationTrainer(model5, fold_loaders, config, device=device)
results = trainer.train_all_folds()

🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features: 4
   Node conv dims: [32, 32, 32, 32]
   Edge conv dims: [32, 32, 32, 32]
   MLP dims: [128, 128]
   Num layers: 4
   Total de parâmetros: 42,306
✅ Modelo construído!


=== Treinando Fold 1 ===
Época 010: Loss=0.6320, Train=0.6311, Val=0.5962, Test=0.5465
Época 020: Loss=0.6165, Train=0.6553, Val=0.5385, Test=0.5581
Época 030: Loss=0.6008, Train=0.6602, Val=0.5000, Test=0.5581
Época 040: Loss=0.5859, Train=0.6796, Val=0.5385, Test=0.5930
Época 050: Loss=0.5663, Train=0.7039, Val=0.5769, Test=0.5814
Época 060: Loss=0.5696, Train=0.6845, Val=0.5577, Test=0.6047
Época 070: Loss=0.5741, Train=0.6553, Val=0.5577, Test=0.5814
Época 080: Loss=0.5574, Train=0.6893, Val=0.5769, Test=0.5698
Época 090: Loss=0.5563, Train=0.7136, Val=0.5577, Test=0.6047
Época 100: Loss=0.5642, Train=0.7379, Val=0.5962, Test=0.5930
Melhor Val Acc: 0.5962
Test Acc final: 0.5930

=== Treinando Fold 2 ===
Época 010: Loss=0.6185, Train

In [16]:
model5 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[32, 32, 32, 32],      # Nós: 3 camadas
    edge_hidden_dims=[32, 32, 32, 32],     # Arestas: dims maiores!
    mlp_hidden_dims=[128, 128],
    dropout=0.01,
    pooling= "max"
)
config = {
    'optimizer': 'adam',
    'lr': 1e-4,
    'weight_decay': 0,
    'scheduler': 'step',
    'step_size': 50,
    'gamma': 0.5,
    'epochs': 100,
    # 'clip_grad': 1.0,
    # 'early_stopping': True,
    # 'patience': 50,
    'log_interval': 10
}

trainer = GraphClassificationTrainer(model5, fold_loaders, config, device=device)
results = trainer.train_all_folds()

🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features: 4
   Node conv dims: [32, 32, 32, 32]
   Edge conv dims: [32, 32, 32, 32]
   MLP dims: [128, 128]
   Num layers: 4
   Total de parâmetros: 42,306
✅ Modelo construído!


=== Treinando Fold 1 ===
Época 010: Loss=0.6603, Train=0.5971, Val=0.5769, Test=0.5698
Época 020: Loss=0.6421, Train=0.6359, Val=0.5962, Test=0.5465
Época 030: Loss=0.6274, Train=0.6650, Val=0.5962, Test=0.5698
Época 040: Loss=0.6124, Train=0.6505, Val=0.6346, Test=0.5698
Época 050: Loss=0.6006, Train=0.6796, Val=0.5769, Test=0.5581
Época 060: Loss=0.5783, Train=0.7136, Val=0.6154, Test=0.5465
Época 070: Loss=0.5746, Train=0.6893, Val=0.5769, Test=0.5349
Época 080: Loss=0.5629, Train=0.6796, Val=0.6538, Test=0.5465
Época 090: Loss=0.5530, Train=0.6990, Val=0.6923, Test=0.5465
Época 100: Loss=0.5505, Train=0.7233, Val=0.6538, Test=0.5465
Melhor Val Acc: 0.6538
Test Acc final: 0.5465

=== Treinando Fold 2 ===
Época 010: Loss=0.6458, Train

In [17]:
model6 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[32, 32, 32, 32, 32],
    edge_hidden_dims=[32, 32, 32, 32, 32],   
    mlp_hidden_dims=[128, 128],
    dropout=0.01,
    pooling= "sum"
)
config = {
    'optimizer': 'adam',
    'lr': 1e-4,
    'weight_decay': 0,
    'scheduler': 'step',
    'step_size': 50,
    'gamma': 0.5,
    'epochs': 150,
    # 'clip_grad': 1.0,
    # 'early_stopping': True,
    # 'patience': 50,
    'log_interval': 10
}

trainer = GraphClassificationTrainer(model5, fold_loaders, config, device=device)
results = trainer.train_all_folds()

🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features: 4
   Node conv dims: [32, 32, 32, 32, 32]
   Edge conv dims: [32, 32, 32, 32, 32]
   MLP dims: [128, 128]
   Num layers: 5
   Total de parâmetros: 46,658
✅ Modelo construído!


=== Treinando Fold 1 ===
Época 010: Loss=0.6642, Train=0.5680, Val=0.5577, Test=0.5581
Época 020: Loss=0.6352, Train=0.6311, Val=0.5385, Test=0.5000
Época 030: Loss=0.6279, Train=0.6019, Val=0.5962, Test=0.5465
Época 040: Loss=0.6061, Train=0.6699, Val=0.6154, Test=0.5349
Época 050: Loss=0.6025, Train=0.6359, Val=0.5577, Test=0.5814
Época 060: Loss=0.5823, Train=0.7087, Val=0.5962, Test=0.5581
Época 070: Loss=0.5796, Train=0.6942, Val=0.6731, Test=0.5814
Época 080: Loss=0.5696, Train=0.6796, Val=0.6538, Test=0.5349
Época 090: Loss=0.5833, Train=0.6408, Val=0.6346, Test=0.5698
Época 100: Loss=0.5673, Train=0.6845, Val=0.6538, Test=0.5349
Época 110: Loss=0.5595, Train=0.7039, Val=0.6538, Test=0.5116
Época 120: Loss=0.5498, Train=0

In [18]:
model6 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[64,64,64,64],
    edge_hidden_dims=[64,64,64,64],   
    mlp_hidden_dims=[128, 128],
    dropout=0.01,
    pooling= "sum"
)
config = {
    'optimizer': 'adam',
    'lr': 1e-4,
    'weight_decay': 0,
    'scheduler': 'step',
    'step_size': 50,
    'gamma': 0.5,
    'epochs': 150,
    # 'clip_grad': 1.0,
    # 'early_stopping': True,
    # 'patience': 50,
    'log_interval': 10
}

trainer = GraphClassificationTrainer(model5, fold_loaders, config, device=device)
results = trainer.train_all_folds()

🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features: 4
   Node conv dims: [64, 64, 64, 64]
   Edge conv dims: [64, 64, 64, 64]
   MLP dims: [128, 128]
   Num layers: 4
   Total de parâmetros: 96,386
✅ Modelo construído!


=== Treinando Fold 1 ===
Época 010: Loss=0.6510, Train=0.5825, Val=0.6154, Test=0.5000
Época 020: Loss=0.6306, Train=0.6117, Val=0.6154, Test=0.5581
Época 030: Loss=0.6190, Train=0.6602, Val=0.6346, Test=0.5465
Época 040: Loss=0.6112, Train=0.6602, Val=0.6731, Test=0.5465
Época 050: Loss=0.5950, Train=0.6845, Val=0.5962, Test=0.5581
Época 060: Loss=0.5802, Train=0.6845, Val=0.6731, Test=0.5233
Época 070: Loss=0.5728, Train=0.6845, Val=0.6154, Test=0.5698
Época 080: Loss=0.5612, Train=0.6845, Val=0.6538, Test=0.5930
Época 090: Loss=0.5577, Train=0.6942, Val=0.6731, Test=0.6163
Época 100: Loss=0.5577, Train=0.6942, Val=0.6731, Test=0.5930
Época 110: Loss=0.5528, Train=0.7087, Val=0.6154, Test=0.5814
Época 120: Loss=0.5507, Train=0.7136, V

In [19]:
model6 = DualMessagePassingHodgeGNN(
    num_node_features=dataset.num_node_features,
    num_edge_features=dataset.num_edge_features,
    num_classes=dataset.num_classes,
    node_hidden_dims=[32,64,128,256],
    edge_hidden_dims=[32,64,128,256],   
    mlp_hidden_dims=[128, 64],
    dropout=0.01,
    pooling= "sum"
)
config = {
    'optimizer': 'adam',
    'lr': 1e-4,
    'weight_decay': 0,
    'scheduler': 'step',
    'step_size': 50,
    'gamma': 0.5,
    'epochs': 150,
    # 'clip_grad': 1.0,
    # 'early_stopping': True,
    # 'patience': 50,
    'log_interval': 10
}

trainer = GraphClassificationTrainer(model5, fold_loaders, config, device=device)
results = trainer.train_all_folds()

🏗️  Construindo DualMessagePassingHodgeGNN...
   Node features: 18
   Edge features: 4
   Node conv dims: [32, 64, 128, 256]
   Edge conv dims: [32, 64, 128, 256]
   MLP dims: [128, 64]
   Num layers: 4
   Total de parâmetros: 339,842
✅ Modelo construído!


=== Treinando Fold 1 ===
Época 010: Loss=0.6562, Train=0.5971, Val=0.5962, Test=0.5814
Época 020: Loss=0.6404, Train=0.6262, Val=0.5962, Test=0.5698
Época 030: Loss=0.6287, Train=0.6165, Val=0.6346, Test=0.5465
Época 040: Loss=0.6191, Train=0.6408, Val=0.5962, Test=0.5698
Época 050: Loss=0.5923, Train=0.6796, Val=0.6346, Test=0.5581
Época 060: Loss=0.5877, Train=0.6650, Val=0.6538, Test=0.5465
Época 070: Loss=0.5880, Train=0.6553, Val=0.6346, Test=0.5698
Época 080: Loss=0.5711, Train=0.6602, Val=0.5962, Test=0.5698
Época 090: Loss=0.5758, Train=0.6699, Val=0.6154, Test=0.5349
Época 100: Loss=0.5536, Train=0.6845, Val=0.6346, Test=0.5465
Época 110: Loss=0.5655, Train=0.6505, Val=0.6538, Test=0.5814
Época 120: Loss=0.5742, Train=0.665